# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Method choice: Random Forest Classifier.")
print()
print("Why it fits my lane: my task is binary classification (is_declining vs")
print("not), same as the baseline. A Random Forest can capture non-linear")
print("interactions between position, impressions, and activity that my simple")
print("weighted-rank baseline rule couldn't -- for example, the OPPOSITE finding")
print("from my signal audit (high volume -> lower CTR) suggests non-linear")
print("relationships exist that a tree-based model can pick up naturally.")
print()
print("I'm avoiding Gradient Boosting for now since Random Forest is more robust")
print("to overfitting with less tuning, a safer choice for a first honest model.")
print()
print("I'll also report permutation importance to interpret which features")
print("actually drive predictions, keeping this readable rather than a black box.")

Method choice: Random Forest Classifier.

Why it fits my lane: my task is binary classification (is_declining vs
not), same as the baseline. A Random Forest can capture non-linear
interactions between position, impressions, and activity that my simple
weighted-rank baseline rule couldn't -- for example, the OPPOSITE finding
from my signal audit (high volume -> lower CTR) suggests non-linear
relationships exist that a tree-based model can pick up naturally.

I'm avoiding Gradient Boosting for now since Random Forest is more robust
to overfitting with less tuning, a safer choice for a first honest model.

I'll also report permutation importance to interpret which features
actually drive predictions, keeping this readable rather than a black box.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

import duckdb
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feat = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           SUM(ga4_pageviews) as total_pageviews,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df().fillna(0)

feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)

print(f"Loaded {len(feat):,} pages across {feat['client_hash_id'].nunique():,} clients\n")

# Split design: GROUPED by client, not random row-level split.
# Why: pages from the same client share business/domain characteristics.
# A random split could put similar pages from the same client in both train
# and test, leaking client-specific patterns and inflating test performance.
# Client-grouped splitting ensures the model is tested on truly unseen clients.

splitter = GroupShuffleSplit(test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_hash_id"]))

train, test = feat.iloc[train_idx], feat.iloc[test_idx]
print(f"Train: {len(train):,} pages, {train['client_hash_id'].nunique():,} clients")
print(f"Test:  {len(test):,} pages, {test['client_hash_id'].nunique():,} clients")

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"\nClient overlap between train/test: {len(overlap)} (should be 0)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 176,738 pages across 47 clients

Train: 133,474 pages, 35 clients
Test:  43,264 pages, 12 clients

Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

feature_cols = ["avg_position", "total_impressions", "avg_engaged_sessions",
                 "total_pageviews", "days_seen"]

X_train, y_train = train[feature_cols], train["is_declining"]
X_test, y_test = test[feature_cols], test["is_declining"]

# --- Baseline (Week 4 rule, same idea): rank by weighted position + activity ---
test = test.copy()
test["baseline_score"] = (
    (1 - test["avg_position"].rank(pct=True)) * 0.6 +
    (test["days_seen"].rank(pct=True)) * 0.4
)
baseline_pred = (test["baseline_score"] > test["baseline_score"].quantile(0.70)).astype(int)

baseline_precision = precision_score(y_test, baseline_pred)
baseline_recall = recall_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)

# --- Model: Random Forest ---
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

model_precision = precision_score(y_test, model_pred)
model_recall = recall_score(y_test, model_pred)
model_f1 = f1_score(y_test, model_pred)

import pandas as pd
comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline (rule)", "Random Forest"],
    "Precision": [baseline_precision, model_precision],
    "Recall": [baseline_recall, model_recall],
    "F1": [baseline_f1, model_f1]
})
print("Model vs Baseline comparison (same test set, same metric):\n")
print(comparison.to_string(index=False))

Model vs Baseline comparison (same test set, same metric):

                Method  Precision   Recall       F1
Week-4 Baseline (rule)   0.185222 0.099199 0.129202
         Random Forest   0.855723 0.888174 0.871646


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# Permutation importance -- which features actually drive the model?
perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("Permutation importance (which features matter):")
print(importance_df.to_string(index=False))
print()

# Error analysis: where does the model get it wrong?
test_results = test.copy()
test_results["predicted"] = model_pred
test_results["actual"] = y_test.values
false_negatives = test_results[(test_results["actual"] == 1) & (test_results["predicted"] == 0)]
false_positives = test_results[(test_results["actual"] == 0) & (test_results["predicted"] == 1)]

print(f"False negatives (missed decliners): {len(false_negatives)}")
print(f"False positives (wrongly flagged): {len(false_positives)}")
print()

print("Interpretation:")
print(f"The model leans heavily on: {importance_df.iloc[0]['feature']} and")
print(f"{importance_df.iloc[1]['feature']} -- these matter most for distinguishing")
print("declining pages. The Random Forest massively outperforms the Week-4 rule")
print("(F1: 0.87 vs 0.13), likely because the rule only used a simple weighted")
print("combination of two features, while the forest captures non-linear")
print("interactions across all five features and their combinations.")
print()
print(f"With {len(false_negatives)} false negatives, the model still misses some")
print("real decliners -- likely pages with unusual patterns not well represented")
print("in training data. This is a meaningfully better starting point than the")
print("baseline, but still needs human review before fully automating action.")

Permutation importance (which features matter):
             feature  importance
   total_impressions    0.235489
        avg_position    0.016173
     total_pageviews    0.015618
avg_engaged_sessions    0.005147
           days_seen    0.000326

False negatives (missed decliners): 2710
False positives (wrongly flagged): 3629

Interpretation:
The model leans heavily on: total_impressions and
avg_position -- these matter most for distinguishing
declining pages. The Random Forest massively outperforms the Week-4 rule
(F1: 0.87 vs 0.13), likely because the rule only used a simple weighted
combination of two features, while the forest captures non-linear
interactions across all five features and their combinations.

With 2710 false negatives, the model still misses some
real decliners -- likely pages with unusual patterns not well represented
in training data. This is a meaningfully better starting point than the
baseline, but still needs human review before fully automating action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.